# Creating Subagents

In [1]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

d:\workspace\AI\langchain-academy\intro-2-langchain\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5")

In [3]:
from langchain.agents import create_agent

subagent_1 = create_agent(
    model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model,
    tools = [square]
)

In [4]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({'messages': [HumanMessage(content=f"Calculate the square of {x}")]})
    return response['messages'][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

main_agent = create_agent(
    model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number."
)

# Test

In [5]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='8f2a3de2-42c8-4424-ab40-7ab934aa072f'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-04T19:04:31.6855729Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5323170500, 'load_duration': 4062147400, 'prompt_eval_count': 381, 'prompt_eval_duration': 133022900, 'eval_count': 101, 'eval_duration': 972704900, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d59e1-d7f2-7a03-98c6-4be9d18b1abb-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': '698cb88f-f396-4cb6-908e-ed2668c73d2a', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 381, 'output_tokens': 101, 'total_tokens': 482}),
              ToolMessage(content="To calculate the square of 456.0, I'll multiply 456 by itself:\n\n456.0 × 456.0 = 207,936\n\n**The sq